# Phase 5e: Best Posting Time
Implementing locked specification with parameter selection amendment.

In [1]:
# CHECKPOINT 1 — Environment & Configuration
print("=== CHECKPOINT 1 START ===")
import pandas as pd
import numpy as np
import yaml
import os
import pytz
from datetime import datetime

with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

time_cutoffs = config.get('time_cutoffs', [])
print(f"time_cutoffs: {time_cutoffs}")
print(f"time_half_life_days currently in config: {config.get('time_half_life_days')}")
print(f"shrinkage_k currently in config: {config.get('shrinkage_k')}")

required_files = [
    '../data/processed/posts_clean.parquet',
    '../data/processed/features_cold.parquet'
]
for p in required_files:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing required file: {p}")
print("Required files verified.")
print("=== CHECKPOINT 1 COMPLETE ===")


=== CHECKPOINT 1 START ===


time_cutoffs: ['2019-01-03', '2019-02-26']
time_half_life_days currently in config: None
shrinkage_k currently in config: None
Required files verified.
=== CHECKPOINT 1 COMPLETE ===


In [2]:
# CHECKPOINT 2 — Data Loading & Preparation
print("=== CHECKPOINT 2 START ===")
posts = pd.read_parquet('../data/processed/posts_clean.parquet')
features = pd.read_parquet('../data/processed/features_cold.parquet')

# Obtain time split from features
features_sub = features.reset_index()[['post_id', 'time_split']]
posts = posts.merge(features_sub, on='post_id', how='inner')

# Ensure timestamps are UTC
posts['posted_at'] = pd.to_datetime(posts['posted_at'], utc=True)
print(f"Timestamps are UTC: {posts['posted_at'].dt.tz == pytz.UTC}")

print(f"Row count: {len(posts)}")
print(f"Pool (category) count: {posts['category'].nunique() + 1}") # +1 for 'all'
print(f"Timestamp range: {posts['posted_at'].min()} to {posts['posted_at'].max()}")

print("\nSchemas:")
print(posts[['post_id', 'username', 'category', 'engagement_rate', 'posted_at', 'time_split']].dtypes)
print("=== CHECKPOINT 2 COMPLETE ===")


=== CHECKPOINT 2 START ===


Timestamps are UTC: False
Row count: 1313584
Pool (category) count: 10
Timestamp range: 2012-02-12 07:07:27+00:00 to 2019-05-15 22:58:53+00:00

Schemas:
post_id                            str
username                           str
category                           str
engagement_rate                float64
posted_at          datetime64[us, UTC]
time_split                         str
dtype: object
=== CHECKPOINT 2 COMPLETE ===


In [3]:
# CHECKPOINT 3 — Relative Engagement & Temporal Features
print("=== CHECKPOINT 3 START ===")
# Use reference_date as established
reference_date = pd.to_datetime('2019-05-15', utc=True)

# Calculate account median engagement (using min_account_posts as in 5d)
valid_posts = posts[(posts['has_followers'] == True) & (posts['engagement_rate'].notnull()) & (posts['er_gt_1'] == False)].copy()

account_counts = valid_posts.groupby('username').size()
valid_accounts = account_counts[account_counts >= config['min_account_posts']].index
filtered_posts = valid_posts[valid_posts['username'].isin(valid_accounts)].copy()

account_medians = filtered_posts.groupby('username')['engagement_rate'].median().rename('account_median_er')
filtered_posts = filtered_posts.merge(account_medians, on='username')

# Remove zero medians
filtered_posts = filtered_posts[filtered_posts['account_median_er'] > 0].copy()

# Calculate relative_engagement
filtered_posts['relative_engagement'] = filtered_posts['engagement_rate'] / filtered_posts['account_median_er']

# Extract UTC day_of_week and hour
# day_of_week: Monday=0, Sunday=6 in pandas dt.dayofweek
filtered_posts['day_of_week'] = filtered_posts['posted_at'].dt.dayofweek
filtered_posts['hour'] = filtered_posts['posted_at'].dt.hour
days_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
filtered_posts['day_name'] = filtered_posts['day_of_week'].map(days_map)

# Calculate age_days
filtered_posts['age_days'] = (reference_date - filtered_posts['posted_at']).dt.total_seconds() / 86400.0

print(f"Total posts with valid relative engagement: {len(filtered_posts)}")
print(filtered_posts[['relative_engagement', 'age_days']].describe())

# Save this base df
base_df = filtered_posts.copy()
print("=== CHECKPOINT 3 COMPLETE ===")


=== CHECKPOINT 3 START ===


Total posts with valid relative engagement: 1312252
       relative_engagement      age_days
count         1.312252e+06  1.312252e+06
mean          1.212300e+00  3.736024e+02
std           3.461630e+00  2.962143e+02
min           0.000000e+00 -9.575579e-01
25%           7.514964e-01  1.584556e+02
50%           1.000000e+00  3.005374e+02
75%           1.321474e+00  5.058586e+02
max           1.239200e+03  2.648703e+03
=== CHECKPOINT 3 COMPLETE ===


In [4]:
# CHECKPOINT 4 — Establish Phase 5e Parameters
print("=== CHECKPOINT 4 START ===")

train_df = base_df[base_df['time_split'] == 'train'].copy()
val_df = base_df[base_df['time_split'] == 'val'].copy()

print(f"Parameter-training data (train) count: {len(train_df)}")
print(f"Internal validation (val) count: {len(val_df)}")
print("Final holdout untouched.")

half_lifes = [14, 28, 56, 90]
shrinkages = [10, 50, 100, 500]

pools = list(base_df['category'].unique()) + ['all']

def get_bucket_scores(df, hl, k):
    # Calculate exponential recency weights
    weights = np.exp(-np.log(2) * df['age_days'] / hl)
    
    # We will compute bucket scores per pool
    # First, precompute weighted engagement
    df_temp = df.copy()
    df_temp['weight'] = weights
    df_temp['weighted_eng'] = df_temp['relative_engagement'] * df_temp['weight']
    
    pool_best = {}
    for pool in pools:
        pool_df = df_temp if pool == 'all' else df_temp[df_temp['category'] == pool]
        if pool_df.empty:
            continue
            
        global_n = pool_df['weight'].sum()
        global_mean = pool_df['weighted_eng'].sum() / global_n if global_n > 0 else 0
        
        grp = pool_df.groupby(['day_name', 'hour'])
        # n = sum of weights in the bucket, m = weighted mean
        bucket_sum_w = grp['weight'].sum()
        bucket_sum_we = grp['weighted_eng'].sum()
        
        bucket_n = bucket_sum_w # Using sum of weights as 'n' is standard for weighted shrinkage
        bucket_m = bucket_sum_we / bucket_sum_w
        
        shrunk_mean = (bucket_n * bucket_m + k * global_mean) / (bucket_n + k)
        
        best_idx = shrunk_mean.idxmax()
        if pd.isna(best_idx):
            continue
        pool_best[pool] = best_idx
    return pool_best

results = []

for hl in half_lifes:
    for k in shrinkages:
        best_buckets = get_bucket_scores(train_df, hl, k)
        
        pool_means = []
        for pool, best_b in best_buckets.items():
            pool_val = val_df if pool == 'all' else val_df[val_df['category'] == pool]
            # realized mean on val
            b_val = pool_val[(pool_val['day_name'] == best_b[0]) & (pool_val['hour'] == best_b[1])]
            if len(b_val) >= 10: # minimum evaluation samples
                pool_means.append(b_val['relative_engagement'].mean())
                
        valid_pools = len(pool_means)
        if valid_pools > 0:
            macro_avg = np.mean(pool_means)
        else:
            macro_avg = -1
            
        results.append({
            'time_half_life_days': hl,
            'shrinkage_k': k,
            'validation_macro_avg': macro_avg,
            'contributing_pool_count': valid_pools,
            'selected/valid': 'Valid' if valid_pools == len(pools) else 'Invalid (too few pools)'
        })

grid_df = pd.DataFrame(results)
print("\nParameter Grid Evaluation:")
print(grid_df.to_string(index=False))

valid_grid = grid_df[grid_df['selected/valid'] == 'Valid']
if valid_grid.empty:
    raise ValueError("No parameter pair has sufficient eligible pools.")
    
best_row = valid_grid.loc[valid_grid['validation_macro_avg'].idxmax()]
best_hl = best_row['time_half_life_days']
best_k = best_row['shrinkage_k']

print("\n=== PARAMETERS FROZEN ===")
print(f"time_half_life_days = {best_hl}")
print(f"shrinkage_k = {best_k}")

config['time_half_life_days'] = int(best_hl)
config['shrinkage_k'] = int(best_k)

with open('../config.yaml', 'w') as f:
    yaml.dump(config, f, sort_keys=True)

print("=== CHECKPOINT 4 COMPLETE ===")


=== CHECKPOINT 4 START ===


Parameter-training data (train) count: 1048192
Internal validation (val) count: 132253
Final holdout untouched.



Parameter Grid Evaluation:
 time_half_life_days  shrinkage_k  validation_macro_avg  contributing_pool_count          selected/valid
                  14           10              1.348969                       10                   Valid
                  14           50              1.348969                       10                   Valid
                  14          100              1.348969                       10                   Valid
                  14          500              1.348969                       10                   Valid
                  28           10              1.316811                       10                   Valid
                  28           50              1.290842                       10                   Valid
                  28          100              1.290842                       10                   Valid
                  28          500              1.290842                       10                   Valid
                  56       

In [5]:
# CHECKPOINT 5 — Bucket Scoring
print("=== CHECKPOINT 5 START ===")

pre_cutoff_df = base_df[base_df['time_split'].isin(['train', 'val'])].copy()
pre_cutoff_df['weight'] = np.exp(-np.log(2) * pre_cutoff_df['age_days'] / best_hl)
pre_cutoff_df['weighted_eng'] = pre_cutoff_df['relative_engagement'] * pre_cutoff_df['weight']

final_buckets = {}
bucket_tables = {}

for pool in pools:
    pool_df = pre_cutoff_df if pool == 'all' else pre_cutoff_df[pre_cutoff_df['category'] == pool]
    
    global_n = pool_df['weight'].sum()
    global_mean = pool_df['weighted_eng'].sum() / global_n if global_n > 0 else 0
    
    grp = pool_df.groupby(['day_name', 'hour'])
    bucket_n = grp['weight'].sum()
    bucket_m = grp['weighted_eng'].sum() / bucket_n
    
    shrunk_mean = (bucket_n * bucket_m + best_k * global_mean) / (bucket_n + best_k)
    
    b_df = pd.DataFrame({'bucket_n': bucket_n, 'bucket_m': bucket_m, 'shrunk_mean': shrunk_mean})
    b_df['raw_count'] = grp.size()
    b_df = b_df.reset_index()
    
    best_idx = b_df['shrunk_mean'].idxmax()
    best_b = b_df.loc[best_idx]
    
    final_buckets[pool] = (best_b['day_name'], int(best_b['hour']))
    bucket_tables[pool] = b_df

print("\nSelected Buckets (Pre-cutoff data):")
for pool, b in final_buckets.items():
    print(f"Pool: {pool:<10} | {b[0]:<10} {b[1]:02d}:00")

print("\nRepresentative Bucket Table (top 5 for 'all'):")
print(bucket_tables['all'].sort_values('shrunk_mean', ascending=False).head())

print("=== CHECKPOINT 5 COMPLETE ===")


=== CHECKPOINT 5 START ===



Selected Buckets (Pre-cutoff data):
Pool: fashion    | Monday     02:00
Pool: other      | Monday     01:00
Pool: food       | Tuesday    21:00
Pool: interior   | Friday     20:00
Pool: family     | Friday     00:00
Pool: fitness    | Thursday   13:00
Pool: beauty     | Tuesday    16:00
Pool: travel     | Sunday     04:00
Pool: pet        | Friday     19:00
Pool: all        | Monday     02:00

Representative Bucket Table (top 5 for 'all'):
    day_name  hour  bucket_n  bucket_m  shrunk_mean  raw_count
26    Monday     2  6.189370  1.697542     1.481767       5748
25    Monday     1  7.292917  1.623234     1.464199       6903
77    Sunday     5  2.161268  1.817035     1.431533       2495
76    Sunday     4  2.667740  1.731161     1.428862       2948
121  Tuesday     1  5.909572  1.548175     1.422490       7458
=== CHECKPOINT 5 COMPLETE ===


In [6]:
# CHECKPOINT 6 — Temporal Holdout Evaluation
print("=== CHECKPOINT 6 START ===")

holdout_df = base_df[base_df['time_split'] == 'test'].copy()
np.random.seed(config['seed'])

print(f"{'Pool':<10} | {'Selected Bucket':<15} | {'L-Mean':<7} | {'L-Most':<7} | {'95% CI (vs overall)'}")
print("-" * 70)

holdout_metrics = []

for pool in pools:
    b_day, b_hour = final_buckets[pool]
    pool_holdout = holdout_df if pool == 'all' else holdout_df[holdout_df['category'] == pool]
    
    if pool_holdout.empty:
        continue
        
    overall_mean = pool_holdout['relative_engagement'].mean()
    
    grp = pool_holdout.groupby(['day_name', 'hour'])
    if len(grp) == 0:
        continue
    most_posted_idx = grp.size().idxmax()
    most_posted_mean = grp['relative_engagement'].mean().loc[most_posted_idx]
    
    selected_vals = pool_holdout[(pool_holdout['day_name'] == b_day) & (pool_holdout['hour'] == b_hour)]['relative_engagement'].values
    if len(selected_vals) == 0:
        continue
    selected_mean = selected_vals.mean()
    
    lift_mean = selected_mean - overall_mean
    lift_most = selected_mean - most_posted_mean
    
    # Bootstrap CI
    n_boot = 1000
    pool_vals = pool_holdout['relative_engagement'].values
    diffs = []
    for _ in range(n_boot):
        b_sel = np.random.choice(selected_vals, size=len(selected_vals), replace=True)
        b_all = np.random.choice(pool_vals, size=len(pool_vals), replace=True)
        diffs.append(np.mean(b_sel) - np.mean(b_all))
    ci = np.percentile(diffs, [2.5, 97.5])
    
    holdout_metrics.append({
        'pool': pool, 'bucket': f"{b_day[:3]} {b_hour:02d}", 
        'lift_mean': lift_mean, 'lift_most': lift_most, 'ci': ci
    })
    
    ci_str = f"[{ci[0]:.4f}, {ci[1]:.4f}]"
    print(f"{pool:<10} | {b_day[:3]} {b_hour:02d}          | {lift_mean:+.4f} | {lift_most:+.4f} | {ci_str}")

print("=== CHECKPOINT 6 COMPLETE ===")


=== CHECKPOINT 6 START ===
Pool       | Selected Bucket | L-Mean  | L-Most  | 95% CI (vs overall)
----------------------------------------------------------------------


fashion    | Mon 02          | +0.2165 | +0.2058 | [-0.0705, 0.6512]


other      | Mon 01          | +0.0649 | -0.0231 | [-0.1703, 0.3504]
food       | Tue 21          | +0.0445 | -0.0816 | [-0.1321, 0.2429]


interior   | Fri 20          | -0.1658 | -0.1007 | [-0.3381, 0.0152]
family     | Fri 00          | +0.2140 | +0.2628 | [-0.1235, 0.7060]
fitness    | Thu 13          | +0.9387 | +0.8768 | [-0.0589, 2.3484]


beauty     | Tue 16          | +0.1892 | +0.1662 | [-0.4358, 1.0596]
travel     | Sun 04          | +0.1026 | +0.0823 | [-0.3522, 0.7695]
pet        | Fri 19          | +0.1004 | +0.2536 | [-0.1405, 0.3655]


all        | Mon 02          | +0.1661 | +0.1563 | [-0.0305, 0.4232]
=== CHECKPOINT 6 COMPLETE ===


In [7]:
# CHECKPOINT 7 — Final Artifact & Inference Validation
print("=== CHECKPOINT 7 START ===")

# Recompute using ALL posts
all_df = base_df.copy()
all_df['weight'] = np.exp(-np.log(2) * all_df['age_days'] / best_hl)
all_df['weighted_eng'] = all_df['relative_engagement'] * all_df['weight']

final_all_buckets = []
for pool in pools:
    pool_df = all_df if pool == 'all' else all_df[all_df['category'] == pool]
    
    global_n = pool_df['weight'].sum()
    global_mean = pool_df['weighted_eng'].sum() / global_n if global_n > 0 else 0
    
    grp = pool_df.groupby(['day_name', 'hour'])
    bucket_n = grp['weight'].sum()
    bucket_m = grp['weighted_eng'].sum() / bucket_n
    
    shrunk_mean = (bucket_n * bucket_m + best_k * global_mean) / (bucket_n + best_k)
    b_df = pd.DataFrame({'shrunk_mean': shrunk_mean}).reset_index()
    b_df['pool'] = pool
    final_all_buckets.append(b_df)
    
artifact_df = pd.concat(final_all_buckets, ignore_index=True)
artifact_df.to_parquet('../artifacts/v1.0/posting_time.parquet', index=False)

# Validation logic for inference
import pytz
from datetime import datetime, timedelta

def get_recommended_posting_time(pool, artifact_df, tz_str="UTC"):
    pool_name = pool if pool in artifact_df['pool'].values else 'all'
    pool_data = artifact_df[artifact_df['pool'] == pool_name]
    best_b = pool_data.loc[pool_data['shrunk_mean'].idxmax()]
    
    day_name = best_b['day_name']
    hour = int(best_b['hour'])
    
    # Timezone conversion using DST on reference date (2019-05-15 UTC)
    ref_dt = datetime(2019, 5, 15, hour, 0, tzinfo=pytz.UTC)
    
    days_map_rev = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
    ref_day_of_week = ref_dt.weekday()
    target_day_of_week = days_map_rev[day_name]
    
    day_diff = (target_day_of_week - ref_day_of_week) % 7
    bucket_utc = ref_dt + timedelta(days=day_diff)
    
    try:
        target_tz = pytz.timezone(tz_str)
    except pytz.UnknownTimeZoneError:
        target_tz = pytz.UTC
        tz_str = "UTC"
        
    bucket_local = bucket_utc.astimezone(target_tz)
    
    days_map_fwd = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
    
    start_str = bucket_local.strftime('%H:%M')
    end_local = bucket_local + timedelta(hours=1)
    end_str = end_local.strftime('%H:%M')
    
    return {
        "day": days_map_fwd[bucket_local.weekday()],
        "start": start_str,
        "end": end_str,
        "timezone": tz_str
    }

print("Inference Test UTC (beauty):")
print(get_recommended_posting_time('beauty', artifact_df, 'UTC'))

print("Inference Test America/New_York (beauty):")
print(get_recommended_posting_time('beauty', artifact_df, 'America/New_York'))

print("=== CHECKPOINT 7 COMPLETE ===")


=== CHECKPOINT 7 START ===


Inference Test UTC (beauty):
{'day': 'Saturday', 'start': '14:00', 'end': '15:00', 'timezone': 'UTC'}
Inference Test America/New_York (beauty):


{'day': 'Saturday', 'start': '10:00', 'end': '11:00', 'timezone': 'America/New_York'}
=== CHECKPOINT 7 COMPLETE ===
